# Wine Quality — Decision Tree and Random Forest
## Project 11

Predict wine quality scores with a Decision Tree and Random Forest classifier.

**Topics:** EDA, missing values, preprocessing, multiclass classification, confusion
matrix, feature importance, model comparison and 5-fold cross-validation.

> The included 1,599-row CSV is a reproducible educational practice dataset using a
> red-wine-quality-style schema. It is not claimed as an official UCI/Kaggle download.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss,confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
BASE=Path.cwd()
if not (BASE/'dataset').exists(): BASE=BASE.parent
DATA=BASE/'dataset'/'winequality_red.csv'; OUT=BASE/'outputs'; OUT.mkdir(exist_ok=True)
df=pd.read_csv(DATA); display(df.head()); print(df.shape)

## 1. EDA and Data Quality

In [ ]:
display(df.describe().T)
display(pd.DataFrame({'Missing_Count':df.isna().sum(),'Missing_Percent':df.isna().mean()*100}))
display(df.quality.value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8,5)); plt.hist(df.alcohol.dropna(),bins=20); plt.title('Alcohol Distribution'); plt.show()
plt.figure(figsize=(8,5)); df.groupby('quality').alcohol.mean().plot(kind='bar'); plt.title('Mean Alcohol by Quality'); plt.show()

## 2. Preprocessing and Split

In [ ]:
X=df.drop(columns='quality'); y=df.quality; features=X.columns.tolist()
pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),features)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)

## 3. Decision Tree

In [ ]:
dt=DecisionTreeClassifier(max_depth=5,min_samples_leaf=8,class_weight='balanced',random_state=42)
dt_pipe=Pipeline([('preprocessor',pre),('model',dt)])
dt_pipe.fit(X_train,y_train); dt_pred=dt_pipe.predict(X_test); dt_prob=dt_pipe.predict_proba(X_test)

## 4. Random Forest

In [ ]:
rf=RandomForestClassifier(n_estimators=250,max_depth=10,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=1)
rf_pipe=Pipeline([('preprocessor',pre),('model',rf)])
rf_pipe.fit(X_train,y_train); rf_pred=rf_pipe.predict(X_test); rf_prob=rf_pipe.predict_proba(X_test)

## 5. Model Comparison

In [ ]:
comparison=pd.DataFrame([
 {'Model':'Decision Tree','Accuracy':accuracy_score(y_test,dt_pred),'Precision_Macro':precision_score(y_test,dt_pred,average='macro',zero_division=0),'Recall_Macro':recall_score(y_test,dt_pred,average='macro',zero_division=0),'F1_Macro':f1_score(y_test,dt_pred,average='macro',zero_division=0),'ROC_AUC_OVR_Macro':roc_auc_score(y_test,dt_prob,multi_class='ovr',average='macro'),'Log_Loss':log_loss(y_test,dt_prob)},
 {'Model':'Random Forest','Accuracy':accuracy_score(y_test,rf_pred),'Precision_Macro':precision_score(y_test,rf_pred,average='macro',zero_division=0),'Recall_Macro':recall_score(y_test,rf_pred,average='macro',zero_division=0),'F1_Macro':f1_score(y_test,rf_pred,average='macro',zero_division=0),'ROC_AUC_OVR_Macro':roc_auc_score(y_test,rf_prob,multi_class='ovr',average='macro'),'Log_Loss':log_loss(y_test,rf_prob)}
])
display(comparison); comparison.to_csv(OUT/'model_comparison_metrics.csv',index=False)

## 6. Confusion Matrix

In [ ]:
best_pred=rf_pred if comparison.loc[comparison.ROC_AUC_OVR_Macro.idxmax(),'Model']=='Random Forest' else dt_pred
display(pd.DataFrame(confusion_matrix(y_test,best_pred,labels=sorted(y.unique())),index=sorted(y.unique()),columns=sorted(y.unique())))

## 7. Feature Importance

In [ ]:
dt_imp=pd.DataFrame({'Feature':features,'Importance':dt_pipe.named_steps['model'].feature_importances_}).sort_values('Importance',ascending=False)
rf_imp=pd.DataFrame({'Feature':features,'Importance':rf_pipe.named_steps['model'].feature_importances_}).sort_values('Importance',ascending=False)
display(dt_imp); display(rf_imp)
dt_imp.to_csv(OUT/'decision_tree_feature_importance.csv',index=False); rf_imp.to_csv(OUT/'random_forest_feature_importance.csv',index=False)

## 8. Five-Fold Cross-Validation

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=42); rows=[]
for name,est in [('Decision Tree',dt),('Random Forest',rf)]:
    acc=[]; f1s=[]; auc=[]
    for tr,va in cv.split(X,y):
        pipe=Pipeline([('preprocessor',pre),('model',est)]); pipe.fit(X.iloc[tr],y.iloc[tr])
        pp=pipe.predict(X.iloc[va]); qq=pipe.predict_proba(X.iloc[va])
        acc.append(accuracy_score(y.iloc[va],pp)); f1s.append(f1_score(y.iloc[va],pp,average='macro',zero_division=0)); auc.append(roc_auc_score(y.iloc[va],qq,multi_class='ovr',average='macro'))
    rows.append({'Model':name,'Accuracy_Mean':np.mean(acc),'F1_Macro_Mean':np.mean(f1s),'ROC_AUC_Macro_Mean':np.mean(auc)})
cv_summary=pd.DataFrame(rows); display(cv_summary); cv_summary.to_csv(OUT/'cross_validation_comparison.csv',index=False)

## 9. Conclusion

Decision Tree is easy to interpret, while Random Forest combines many trees to reduce
variance and generally provides stronger ensemble performance.

This project is for educational machine-learning practice only.